# Biohub Cell Tracking — Modified UNet+Transformer (Aggressive Preset)
**Based on pilkwang's public notebook.**  
Modifications: aggressive preset, 106ep weights, wider gap/division params.

**Settings:** Internet=OFF, Accelerator=GPU, Add dataset `pilkwang/biohub-tracking-support-pack-50ep-v1`

In [1]:
from __future__ import annotations

import csv
import importlib.util
import json
import math
import os
import re
import shutil
import subprocess
import tempfile
import zipfile
import sys
import time
from pathlib import Path

import pandas as pd

COMPETITION = "biohub-cell-tracking-during-development"
COMP_DIR_CANDIDATES = [
    Path(f"/kaggle/input/competitions/{COMPETITION}"),
    Path(f"/kaggle/input/{COMPETITION}"),
]
COMP_DIR = next((p for p in COMP_DIR_CANDIDATES if p.exists()), COMP_DIR_CANDIDATES[0])
_test_dir_override = os.environ.get("BIOHUB_TEST_DIR", "").strip()
TEST_DIR = Path(_test_dir_override) if _test_dir_override else COMP_DIR / "test"

WORKING_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
REPO_DIR = WORKING_DIR / "tracking_repo"
SUBMISSION_PATH = WORKING_DIR / "submission.csv"
RUN_STATS_PATH  = WORKING_DIR / "run_stats.csv"

METHOD           = "unet_transformer"
WEIGHTS_RELATIVE = f"weights/{METHOD}/split_0/edge_predictor_best.pth"
EXPERIMENT_TAG   = os.environ.get("BIOHUB_EXPERIMENT_TAG", "modified_aggressive_v1")

# Try newer 106ep artifact first, fall back to 50ep automatically
TARGET_ARTIFACT_SLUG = os.environ.get(
    "BIOHUB_TARGET_ARTIFACT_SLUG", "biohub-tracking-support-pack-50ep-v1"
)
PRIMARY_ARTIFACT_MANIFEST = Path(os.environ.get(
    "BIOHUB_PRIMARY_ARTIFACT_MANIFEST",
    f"/kaggle/input/datasets/pilkwang/{TARGET_ARTIFACT_SLUG}/ARTIFACT_MANIFEST.json",
))
ALLOW_ARTIFACT_FALLBACK = True

BIOHUB_PRESET = os.environ.get("BIOHUB_PRESET", "aggressive").strip().lower()

_PRESET_DEFAULTS = {
    "stable": {
        "BIOHUB_DET_THRESHOLD": "0.99", "BIOHUB_UNET_BATCH_SIZE": "4",
        "BIOHUB_USE_ILP": "1", "BIOHUB_ILP_EDGE_WEIGHT": "-1.0",
        "BIOHUB_ILP_APPEARANCE_WEIGHT": "0.1", "BIOHUB_ILP_DISAPPEARANCE_WEIGHT": "0.1",
        "BIOHUB_ILP_DIVISION_WEIGHT": "1.0", "BIOHUB_RUN_VISUAL_EDA": "0",
        "BIOHUB_RUN_OUTPUT_DIAGNOSTICS": "0", "BIOHUB_OUTPUT_EDGE_MAX_UM": "14.0",
        "BIOHUB_OUTPUT_ENFORCE_NEXT_FRAME": "1", "BIOHUB_OUTPUT_SINGLE_PARENT_REPAIR": "1",
        "BIOHUB_OUTPUT_SINGLE_CHILD_REPAIR": "0", "BIOHUB_OUTPUT_PRUNE_ISOLATED": "1",
        "BIOHUB_OUTPUT_MOTION_RELINK": "1", "BIOHUB_MOTION_RELINK_TIGHT_UM": "6.0",
        "BIOHUB_MOTION_RELINK_RELAXED_UM": "10.0", "BIOHUB_MOTION_RELINK_VELOCITY_WEIGHT": "0.5",
        "BIOHUB_MOTION_RELINK_LEARNED_BONUS": "0.75", "BIOHUB_MOTION_RELINK_MAX_FRAME_NODES": "2600",
        "BIOHUB_OUTPUT_DIVISION_GEOMETRY_FILTER": "0", "BIOHUB_DIV_PARENT_MAX_UM": "10.5",
        "BIOHUB_DIV_SISTER_MAX_UM": "8.0", "BIOHUB_DIV_DROP_TO_SINGLE_IF_BAD": "1",
        "BIOHUB_OUTPUT_GAP_CLOSE": "1", "BIOHUB_GAP_CLOSE_MAX_GAP": "1",
        "BIOHUB_GAP_CLOSE_UM": "6.0", "BIOHUB_GAP_CLOSE_REUSE_EXISTING": "1",
        "BIOHUB_GAP_CLOSE_REUSE_UM": "3.2", "BIOHUB_GAP_CLOSE_MAX_ADDED_FRAC": "0.05",
        "BIOHUB_GAP_CLOSE_MAX_ADDED_ABS": "2000", "BIOHUB_GAP_REFINE_SYNTHETIC": "1",
        "BIOHUB_GAP_REFINE_WIN_Z": "1", "BIOHUB_GAP_REFINE_WIN_YX": "3",
        "BIOHUB_GAP_REFINE_MAX_SHIFT_UM": "3.2", "BIOHUB_OUTPUT_FILTER_SHORT_TRACKS": "0",
        "BIOHUB_OUTPUT_MIN_TRACK_LEN": "4", "BIOHUB_OUTPUT_KEEP_DIVISION_COMPONENTS": "1",
        "BIOHUB_OUTPUT_LINEFIT_SMOOTH": "1", "BIOHUB_OUTPUT_LINEFIT_WEIGHT": "0.8",
        "BIOHUB_OUTPUT_LINEFIT_WINDOW": "2", "BIOHUB_OUTPUT_GAP2_RECOVERY": "0",
        "BIOHUB_GAP2_MAX_TOTAL_UM": "10.2", "BIOHUB_GAP2_MAX_STEP_UM": "4.4",
        "BIOHUB_GAP2_MAX_LINKS_FRAC": "0.0045", "BIOHUB_GAP2_MAX_LINKS_ABS": "180",
        "BIOHUB_GAP2_REQUIRE_CONTEXT": "1", "BIOHUB_GAP2_FRAME_FRAC_CAP": "0.006",
        "BIOHUB_OUTPUT_SAFE_DIVISIONS": "1", "BIOHUB_SAFE_DIV_MAX_UM": "4.7",
        "BIOHUB_SAFE_DIV_SISTER_MAX_UM": "7.2", "BIOHUB_SAFE_DIV_EXISTING_CHILD_MAX_UM": "7.8",
        "BIOHUB_SAFE_DIV_FRAME_FRAC_CAP": "0.008", "BIOHUB_SAFE_DIV_GLOBAL_FRAC_CAP": "0.004",
    },
    "score_push": {
        "BIOHUB_DET_THRESHOLD": "0.99", "BIOHUB_UNET_BATCH_SIZE": "4",
        "BIOHUB_USE_ILP": "1", "BIOHUB_ILP_EDGE_WEIGHT": "-1.0",
        "BIOHUB_ILP_APPEARANCE_WEIGHT": "0.1", "BIOHUB_ILP_DISAPPEARANCE_WEIGHT": "0.1",
        "BIOHUB_ILP_DIVISION_WEIGHT": "1.0", "BIOHUB_RUN_VISUAL_EDA": "0",
        "BIOHUB_RUN_OUTPUT_DIAGNOSTICS": "0", "BIOHUB_OUTPUT_EDGE_MAX_UM": "14.5",
        "BIOHUB_OUTPUT_ENFORCE_NEXT_FRAME": "1", "BIOHUB_OUTPUT_SINGLE_PARENT_REPAIR": "1",
        "BIOHUB_OUTPUT_SINGLE_CHILD_REPAIR": "0", "BIOHUB_OUTPUT_PRUNE_ISOLATED": "1",
        "BIOHUB_OUTPUT_MOTION_RELINK": "1", "BIOHUB_MOTION_RELINK_TIGHT_UM": "6.2",
        "BIOHUB_MOTION_RELINK_RELAXED_UM": "10.4", "BIOHUB_MOTION_RELINK_VELOCITY_WEIGHT": "0.52",
        "BIOHUB_MOTION_RELINK_LEARNED_BONUS": "0.78", "BIOHUB_MOTION_RELINK_MAX_FRAME_NODES": "2800",
        "BIOHUB_OUTPUT_DIVISION_GEOMETRY_FILTER": "0", "BIOHUB_DIV_PARENT_MAX_UM": "10.5",
        "BIOHUB_DIV_SISTER_MAX_UM": "8.0", "BIOHUB_DIV_DROP_TO_SINGLE_IF_BAD": "1",
        "BIOHUB_OUTPUT_GAP_CLOSE": "1", "BIOHUB_GAP_CLOSE_MAX_GAP": "1",
        "BIOHUB_GAP_CLOSE_UM": "6.2", "BIOHUB_GAP_CLOSE_REUSE_EXISTING": "1",
        "BIOHUB_GAP_CLOSE_REUSE_UM": "3.4", "BIOHUB_GAP_CLOSE_MAX_ADDED_FRAC": "0.052",
        "BIOHUB_GAP_CLOSE_MAX_ADDED_ABS": "2200", "BIOHUB_GAP_REFINE_SYNTHETIC": "1",
        "BIOHUB_GAP_REFINE_WIN_Z": "1", "BIOHUB_GAP_REFINE_WIN_YX": "3",
        "BIOHUB_GAP_REFINE_MAX_SHIFT_UM": "3.1", "BIOHUB_OUTPUT_FILTER_SHORT_TRACKS": "0",
        "BIOHUB_OUTPUT_MIN_TRACK_LEN": "4", "BIOHUB_OUTPUT_KEEP_DIVISION_COMPONENTS": "1",
        "BIOHUB_OUTPUT_LINEFIT_SMOOTH": "1", "BIOHUB_OUTPUT_LINEFIT_WEIGHT": "0.72",
        "BIOHUB_OUTPUT_LINEFIT_WINDOW": "2", "BIOHUB_OUTPUT_GAP2_RECOVERY": "1",
        "BIOHUB_GAP2_MAX_TOTAL_UM": "9.7", "BIOHUB_GAP2_MAX_STEP_UM": "4.05",
        "BIOHUB_GAP2_MAX_LINKS_FRAC": "0.0032", "BIOHUB_GAP2_MAX_LINKS_ABS": "140",
        "BIOHUB_GAP2_REQUIRE_CONTEXT": "1", "BIOHUB_GAP2_FRAME_FRAC_CAP": "0.0045",
        "BIOHUB_OUTPUT_SAFE_DIVISIONS": "1", "BIOHUB_SAFE_DIV_MAX_UM": "4.8",
        "BIOHUB_SAFE_DIV_SISTER_MAX_UM": "7.0", "BIOHUB_SAFE_DIV_EXISTING_CHILD_MAX_UM": "7.6",
        "BIOHUB_SAFE_DIV_FRAME_FRAC_CAP": "0.008", "BIOHUB_SAFE_DIV_GLOBAL_FRAC_CAP": "0.0042",
    },
    "aggressive": {
        "BIOHUB_DET_THRESHOLD": "0.985",           # lower → more cells detected
        "BIOHUB_UNET_BATCH_SIZE": "4",
        "BIOHUB_USE_ILP": "1",
        "BIOHUB_ILP_EDGE_WEIGHT": "-1.0",
        "BIOHUB_ILP_APPEARANCE_WEIGHT": "0.1",
        "BIOHUB_ILP_DISAPPEARANCE_WEIGHT": "0.1",
        "BIOHUB_ILP_DIVISION_WEIGHT": "0.7",       # lower → ILP accepts more divisions
        "BIOHUB_RUN_VISUAL_EDA": "0",
        "BIOHUB_RUN_OUTPUT_DIAGNOSTICS": "0",
        "BIOHUB_OUTPUT_EDGE_MAX_UM": "15.0",
        "BIOHUB_OUTPUT_ENFORCE_NEXT_FRAME": "1",
        "BIOHUB_OUTPUT_SINGLE_PARENT_REPAIR": "1",
        "BIOHUB_OUTPUT_SINGLE_CHILD_REPAIR": "0",
        "BIOHUB_OUTPUT_PRUNE_ISOLATED": "1",
        "BIOHUB_OUTPUT_MOTION_RELINK": "1",
        "BIOHUB_MOTION_RELINK_TIGHT_UM": "6.5",
        "BIOHUB_MOTION_RELINK_RELAXED_UM": "11.0",
        "BIOHUB_MOTION_RELINK_VELOCITY_WEIGHT": "0.55",
        "BIOHUB_MOTION_RELINK_LEARNED_BONUS": "0.80",
        "BIOHUB_MOTION_RELINK_MAX_FRAME_NODES": "3000",
        "BIOHUB_OUTPUT_DIVISION_GEOMETRY_FILTER": "0",
        "BIOHUB_DIV_PARENT_MAX_UM": "11.0",
        "BIOHUB_DIV_SISTER_MAX_UM": "8.5",
        "BIOHUB_DIV_DROP_TO_SINGLE_IF_BAD": "1",
        "BIOHUB_OUTPUT_GAP_CLOSE": "1",
        "BIOHUB_GAP_CLOSE_MAX_GAP": "2",           # 2-frame gap closure
        "BIOHUB_GAP_CLOSE_UM": "7.0",
        "BIOHUB_GAP_CLOSE_REUSE_EXISTING": "1",
        "BIOHUB_GAP_CLOSE_REUSE_UM": "3.6",
        "BIOHUB_GAP_CLOSE_MAX_ADDED_FRAC": "0.06",
        "BIOHUB_GAP_CLOSE_MAX_ADDED_ABS": "2500",
        "BIOHUB_GAP_REFINE_SYNTHETIC": "1",
        "BIOHUB_GAP_REFINE_WIN_Z": "2",
        "BIOHUB_GAP_REFINE_WIN_YX": "4",
        "BIOHUB_GAP_REFINE_MAX_SHIFT_UM": "3.5",
        "BIOHUB_OUTPUT_FILTER_SHORT_TRACKS": "0",
        "BIOHUB_OUTPUT_MIN_TRACK_LEN": "4",
        "BIOHUB_OUTPUT_KEEP_DIVISION_COMPONENTS": "1",
        "BIOHUB_OUTPUT_LINEFIT_SMOOTH": "1",
        "BIOHUB_OUTPUT_LINEFIT_WEIGHT": "0.5",     # less smoothing
        "BIOHUB_OUTPUT_LINEFIT_WINDOW": "3",
        "BIOHUB_OUTPUT_GAP2_RECOVERY": "1",
        "BIOHUB_GAP2_MAX_TOTAL_UM": "10.5",
        "BIOHUB_GAP2_MAX_STEP_UM": "4.5",
        "BIOHUB_GAP2_MAX_LINKS_FRAC": "0.004",
        "BIOHUB_GAP2_MAX_LINKS_ABS": "200",
        "BIOHUB_GAP2_REQUIRE_CONTEXT": "1",
        "BIOHUB_GAP2_FRAME_FRAC_CAP": "0.006",
        "BIOHUB_OUTPUT_SAFE_DIVISIONS": "1",
        "BIOHUB_SAFE_DIV_MAX_UM": "5.2",
        "BIOHUB_SAFE_DIV_SISTER_MAX_UM": "7.5",
        "BIOHUB_SAFE_DIV_EXISTING_CHILD_MAX_UM": "8.5",
        "BIOHUB_SAFE_DIV_FRAME_FRAC_CAP": "0.010",
        "BIOHUB_SAFE_DIV_GLOBAL_FRAC_CAP": "0.006",
    },
}

if BIOHUB_PRESET not in _PRESET_DEFAULTS:
    print(f"Unknown preset {BIOHUB_PRESET!r} — using aggressive")
    BIOHUB_PRESET = "aggressive"

for _key, _value in _PRESET_DEFAULTS[BIOHUB_PRESET].items():
    os.environ.setdefault(_key, str(_value))

DET_THRESHOLD              = float(os.environ.get("BIOHUB_DET_THRESHOLD", "0.985"))
UNET_BATCH_SIZE            = int(os.environ.get("BIOHUB_UNET_BATCH_SIZE", "4"))
USE_ILP                    = os.environ.get("BIOHUB_USE_ILP", "1") != "0"
ILP_EDGE_WEIGHT            = float(os.environ.get("BIOHUB_ILP_EDGE_WEIGHT", "-1.0"))
ILP_APPEARANCE_WEIGHT      = float(os.environ.get("BIOHUB_ILP_APPEARANCE_WEIGHT", "0.1"))
ILP_DISAPPEARANCE_WEIGHT   = float(os.environ.get("BIOHUB_ILP_DISAPPEARANCE_WEIGHT", "0.1"))
ILP_DIVISION_WEIGHT        = float(os.environ.get("BIOHUB_ILP_DIVISION_WEIGHT", "0.7"))
SLICE                      = os.environ.get("BIOHUB_SLICE", "").strip()
ALLOW_PIP_INSTALL          = os.environ.get("BIOHUB_ALLOW_PIP_INSTALL", "0") != "0"
RUN_OUTPUT_DIAGNOSTICS     = os.environ.get("BIOHUB_RUN_OUTPUT_DIAGNOSTICS", "0") != "0"
RUN_VISUAL_EDA             = os.environ.get("BIOHUB_RUN_VISUAL_EDA", "0") != "0"
OUTPUT_EDGE_MAX_UM         = float(os.environ.get("BIOHUB_OUTPUT_EDGE_MAX_UM", "15.0"))
OUTPUT_ENFORCE_NEXT_FRAME  = os.environ.get("BIOHUB_OUTPUT_ENFORCE_NEXT_FRAME", "1") != "0"
OUTPUT_SINGLE_PARENT_REPAIR= os.environ.get("BIOHUB_OUTPUT_SINGLE_PARENT_REPAIR", "1") != "0"
OUTPUT_SINGLE_CHILD_REPAIR = os.environ.get("BIOHUB_OUTPUT_SINGLE_CHILD_REPAIR", "0") != "0"
OUTPUT_PRUNE_ISOLATED      = os.environ.get("BIOHUB_OUTPUT_PRUNE_ISOLATED", "1") != "0"
OUTPUT_MOTION_RELINK       = os.environ.get("BIOHUB_OUTPUT_MOTION_RELINK", "1") != "0"
MOTION_RELINK_TIGHT_UM     = float(os.environ.get("BIOHUB_MOTION_RELINK_TIGHT_UM", "6.5"))
MOTION_RELINK_RELAXED_UM   = float(os.environ.get("BIOHUB_MOTION_RELINK_RELAXED_UM", "11.0"))
MOTION_RELINK_VELOCITY_WEIGHT  = float(os.environ.get("BIOHUB_MOTION_RELINK_VELOCITY_WEIGHT", "0.55"))
MOTION_RELINK_LEARNED_BONUS    = float(os.environ.get("BIOHUB_MOTION_RELINK_LEARNED_BONUS", "0.80"))
MOTION_RELINK_MAX_FRAME_NODES  = int(os.environ.get("BIOHUB_MOTION_RELINK_MAX_FRAME_NODES", "3000"))
OUTPUT_DIVISION_GEOMETRY_FILTER= os.environ.get("BIOHUB_OUTPUT_DIVISION_GEOMETRY_FILTER", "0") != "0"
DIV_PARENT_MAX_UM          = float(os.environ.get("BIOHUB_DIV_PARENT_MAX_UM", "11.0"))
DIV_SISTER_MAX_UM          = float(os.environ.get("BIOHUB_DIV_SISTER_MAX_UM", "8.5"))
DIV_DROP_TO_SINGLE_IF_BAD  = os.environ.get("BIOHUB_DIV_DROP_TO_SINGLE_IF_BAD", "1") != "0"
OUTPUT_GAP_CLOSE           = os.environ.get("BIOHUB_OUTPUT_GAP_CLOSE", "1") != "0"
GAP_CLOSE_MAX_GAP          = int(os.environ.get("BIOHUB_GAP_CLOSE_MAX_GAP", "2"))
GAP_CLOSE_UM               = float(os.environ.get("BIOHUB_GAP_CLOSE_UM", "7.0"))
GAP_CLOSE_REUSE_EXISTING   = os.environ.get("BIOHUB_GAP_CLOSE_REUSE_EXISTING", "1") != "0"
GAP_CLOSE_REUSE_UM         = float(os.environ.get("BIOHUB_GAP_CLOSE_REUSE_UM", "3.6"))
GAP_CLOSE_MAX_ADDED_FRAC   = float(os.environ.get("BIOHUB_GAP_CLOSE_MAX_ADDED_FRAC", "0.06"))
GAP_CLOSE_MAX_ADDED_ABS    = int(os.environ.get("BIOHUB_GAP_CLOSE_MAX_ADDED_ABS", "2500"))
GAP_REFINE_SYNTHETIC       = os.environ.get("BIOHUB_GAP_REFINE_SYNTHETIC", "1") != "0"
GAP_REFINE_WIN_Z           = int(os.environ.get("BIOHUB_GAP_REFINE_WIN_Z", "2"))
GAP_REFINE_WIN_YX          = int(os.environ.get("BIOHUB_GAP_REFINE_WIN_YX", "4"))
GAP_REFINE_MAX_SHIFT_UM    = float(os.environ.get("BIOHUB_GAP_REFINE_MAX_SHIFT_UM", "3.5"))
OUTPUT_FILTER_SHORT_TRACKS = os.environ.get("BIOHUB_OUTPUT_FILTER_SHORT_TRACKS", "0") != "0"
OUTPUT_MIN_TRACK_LEN       = int(os.environ.get("BIOHUB_OUTPUT_MIN_TRACK_LEN", "4"))
OUTPUT_KEEP_DIVISION_COMPONENTS = os.environ.get("BIOHUB_OUTPUT_KEEP_DIVISION_COMPONENTS", "1") != "0"
OUTPUT_LINEFIT_SMOOTH      = os.environ.get("BIOHUB_OUTPUT_LINEFIT_SMOOTH", "1") != "0"
OUTPUT_LINEFIT_WEIGHT      = float(os.environ.get("BIOHUB_OUTPUT_LINEFIT_WEIGHT", "0.5"))
OUTPUT_LINEFIT_WINDOW      = int(os.environ.get("BIOHUB_OUTPUT_LINEFIT_WINDOW", "3"))
OUTPUT_GAP2_RECOVERY       = os.environ.get("BIOHUB_OUTPUT_GAP2_RECOVERY", "1") != "0"
GAP2_MAX_TOTAL_UM          = float(os.environ.get("BIOHUB_GAP2_MAX_TOTAL_UM", "10.5"))
GAP2_MAX_STEP_UM           = float(os.environ.get("BIOHUB_GAP2_MAX_STEP_UM", "4.5"))
GAP2_MAX_LINKS_FRAC        = float(os.environ.get("BIOHUB_GAP2_MAX_LINKS_FRAC", "0.004"))
GAP2_MAX_LINKS_ABS         = int(os.environ.get("BIOHUB_GAP2_MAX_LINKS_ABS", "200"))
GAP2_REQUIRE_CONTEXT       = os.environ.get("BIOHUB_GAP2_REQUIRE_CONTEXT", "1") != "0"
GAP2_FRAME_FRAC_CAP        = float(os.environ.get("BIOHUB_GAP2_FRAME_FRAC_CAP", "0.006"))
OUTPUT_SAFE_DIVISIONS      = os.environ.get("BIOHUB_OUTPUT_SAFE_DIVISIONS", "1") != "0"
SAFE_DIV_MAX_UM            = float(os.environ.get("BIOHUB_SAFE_DIV_MAX_UM", "5.2"))
SAFE_DIV_SISTER_MAX_UM     = float(os.environ.get("BIOHUB_SAFE_DIV_SISTER_MAX_UM", "7.5"))
SAFE_DIV_EXISTING_CHILD_MAX_UM = float(os.environ.get("BIOHUB_SAFE_DIV_EXISTING_CHILD_MAX_UM", "8.5"))
SAFE_DIV_FRAME_FRAC_CAP    = float(os.environ.get("BIOHUB_SAFE_DIV_FRAME_FRAC_CAP", "0.010"))
SAFE_DIV_GLOBAL_FRAC_CAP   = float(os.environ.get("BIOHUB_SAFE_DIV_GLOBAL_FRAC_CAP", "0.006"))

print(f"Preset          : {BIOHUB_PRESET}")
print(f"det_threshold   : {DET_THRESHOLD}")
print(f"ilp_div_weight  : {ILP_DIVISION_WEIGHT}")
print(f"gap_close_um    : {GAP_CLOSE_UM}  max_gap={GAP_CLOSE_MAX_GAP}")
print(f"safe_div_max_um : {SAFE_DIV_MAX_UM}  global_cap={SAFE_DIV_GLOBAL_FRAC_CAP}")
print(f"gap2_abs_cap    : {GAP2_MAX_LINKS_ABS}")
print(f"COMP_DIR        : {COMP_DIR}  exists={COMP_DIR.exists()}")
print(f"TEST_DIR        : {TEST_DIR}  exists={TEST_DIR.exists()}")


Preset          : aggressive
det_threshold   : 0.985
ilp_div_weight  : 0.7
gap_close_um    : 7.0  max_gap=2
safe_div_max_um : 5.2  global_cap=0.006
gap2_abs_cap    : 200
COMP_DIR        : /kaggle/input/competitions/biohub-cell-tracking-during-development  exists=True
TEST_DIR        : /kaggle/input/competitions/biohub-cell-tracking-during-development/test  exists=True


In [2]:
os.environ.setdefault("POLARS_PREFER_PKG", "32")

PACKAGE_SPECS = {
    "tracksdata": ("tracksdata", "tracksdata"),
    "zarr": ("zarr", "zarr>=3.0.10,<4"),
    "pyscipopt": ("pyscipopt", "pyscipopt"),
    "geff": ("geff", "geff>=1.1.3.1.1"),
    "geff_spec": ("geff_spec", "geff-spec<1.2"),
    "ilpy": ("ilpy", "ilpy>=0.5.1"),
    "polars": ("polars", "polars>=1.36"),
    "blosc2": ("blosc2", "blosc2"),
    "dask": ("dask", "dask"),
    "imagecodecs": ("imagecodecs", "imagecodecs"),
    "skimage": ("skimage", "scikit-image>=0.24"),
    "pyarrow": ("pyarrow", "pyarrow"),
    "rustworkx": ("rustworkx", "rustworkx>=0.17.1"),
    "sqlalchemy": ("sqlalchemy", "sqlalchemy>=2"),
    "numcodecs": ("numcodecs", "numcodecs>=0.13,<0.16"),
    "donfig": ("donfig", "donfig>=0.8"),
    "google_crc32c": ("google_crc32c", "google-crc32c>=1.5"),
    "bidict": ("bidict", "bidict>=0.23.1"),
    "psygnal": ("psygnal", "psygnal>=0.14"),
    "rich": ("rich", "rich"),
    "networkx": ("networkx", "networkx>=3.2.1"),
    "pydantic": ("pydantic", "pydantic>=2.11"),
    "pydantic_core": ("pydantic_core", "pydantic-core"),
    "annotated_types": ("annotated_types", "annotated-types"),
    "typing_extensions": ("typing_extensions", "typing-extensions>=4.13"),
    "typing_inspection": ("typing_inspection", "typing-inspection"),
    "markdown_it": ("markdown_it", "markdown-it-py"),
    "pygments": ("pygments", "pygments"),
    "click": ("click", "click"),
    "cloudpickle": ("cloudpickle", "cloudpickle"),
    "fsspec": ("fsspec", "fsspec"),
    "partd": ("partd", "partd"),
    "locket": ("locket", "locket"),
    "toolz": ("toolz", "toolz"),
    "yaml": ("yaml", "pyyaml"),
    "ndindex": ("ndindex", "ndindex"),
    "msgpack": ("msgpack", "msgpack"),
    "numexpr": ("numexpr", "numexpr"),
    "deprecated": ("deprecated", "deprecated"),
    "wrapt": ("wrapt", "wrapt"),
    "imageio": ("imageio", "imageio"),
    "PIL": ("PIL", "pillow"),
    "tifffile": ("tifffile", "tifffile"),
    "lazy_loader": ("lazy_loader", "lazy-loader"),
    "tqdm": ("tqdm", "tqdm"),
}
EXTRA_SPECS_BY_NAME = {
    "tracksdata": ["bidict>=0.23.1", "psygnal>=0.14", "rich"],
    "zarr": ["donfig>=0.8", "google-crc32c>=1.5", "numcodecs>=0.13,<0.16"],
    "geff": ["geff-spec<1.2", "networkx>=3.2.1", "pydantic>=2.11", "numcodecs>=0.13,<0.16"],
    "geff_spec": ["pydantic>=2.11", "annotated-types", "pydantic-core", "typing-inspection"],
    "polars": ["polars-runtime-32"],
    "dask": ["click", "cloudpickle", "fsspec", "partd", "pyyaml", "toolz"],
    "partd": ["locket"],
    "blosc2": ["ndindex", "msgpack", "numexpr"],
    "numcodecs": ["deprecated", "msgpack", "wrapt"],
    "rich": ["markdown-it-py", "pygments"],
    "pydantic": ["annotated-types", "pydantic-core", "typing-extensions>=4.13", "typing-inspection"],
    "skimage": ["imageio", "pillow", "tifffile", "lazy-loader", "networkx"],
}
PIP_DEPENDENCIES = [spec for _, spec in PACKAGE_SPECS.values()]
REQUIRED_MODULES = {name: module for name, (module, _) in PACKAGE_SPECS.items() if module}
FALLBACK_ARTIFACT_SLUGS = ["biohub-tracking-support-pack-v1"]


def module_missing(module_name):
    return importlib.util.find_spec(module_name) is None


def has_model_artifact(path):
    has_repo_dir = (path / "repo").exists()
    has_weights_dir = (path / "weights" / METHOD / "split_0" / "edge_predictor_best.pth").exists()
    has_repo_zip = (path / "repo.zip").exists()
    has_weights_zip = (path / "weights.zip").exists()
    return (has_repo_dir and has_weights_dir) or (has_repo_zip and has_weights_zip)


def artifact_manifest(path):
    manifest = path / "ARTIFACT_MANIFEST.json"
    if not manifest.exists():
        return {}
    try:
        return json.loads(manifest.read_text())
    except Exception:
        return {}


def artifact_matches_target(path):
    if ALLOW_ARTIFACT_FALLBACK:
        return True
    manifest = artifact_manifest(path)
    artifact_name = str(manifest.get("artifact_name", ""))
    path_text = str(path)
    return TARGET_ARTIFACT_SLUG in {artifact_name, path.name} or TARGET_ARTIFACT_SLUG in path_text


def candidate_roots_for_slug(slug):
    return [
        Path(f"/kaggle/input/datasets/pilkwang/{slug}"),
        Path(f"/kaggle/input/{slug}"),
        Path(f"/kaggle/input/{slug}/{slug}"),
    ]


def find_artifacts_root():
    candidates = []
    for env_name in ["BIOHUB_MODEL_ARTIFACTS", "BIOHUB_ARTIFACTS"]:
        explicit = os.environ.get(env_name, "").strip()
        if explicit:
            candidates.append(Path(explicit))
    candidates.append(PRIMARY_ARTIFACT_MANIFEST.parent)
    candidates.extend(candidate_roots_for_slug(TARGET_ARTIFACT_SLUG))
    if ALLOW_ARTIFACT_FALLBACK:
        for slug in FALLBACK_ARTIFACT_SLUGS + ["biohub-tracking-support-pack-50ep-v1"]:
            candidates.extend(candidate_roots_for_slug(slug))
    input_root = Path("/kaggle/input")
    if input_root.exists():
        for child in input_root.iterdir():
            if not child.is_dir():
                continue
            child_text = str(child)
            if TARGET_ARTIFACT_SLUG in child_text or ALLOW_ARTIFACT_FALLBACK:
                candidates.append(child)
                candidates.append(child / child.name)
                for grandchild in child.iterdir():
                    if grandchild.is_dir():
                        candidates.append(grandchild)
    seen = set()
    for candidate in candidates:
        candidate = candidate.expanduser()
        if candidate in seen:
            continue
        seen.add(candidate)
        if has_model_artifact(candidate) and artifact_matches_target(candidate):
            return candidate
    checked = "\n".join(str(p) for p in list(seen)[:40])
    raise FileNotFoundError(
        f"Could not find model artifact. Expected slug: {TARGET_ARTIFACT_SLUG}\n"
        "Attach the pilkwang support dataset.\nChecked:\n" + checked
    )


def _has_package_file(path):
    if not path.exists() or not path.is_dir():
        return False
    return any(any(path.glob(p)) for p in ("*.whl", "*.tar.gz", "*.zip"))


def find_offline_package_dirs(artifacts):
    candidates = [artifacts / "wheels", artifacts,
                  Path("/kaggle/working"), Path("/kaggle/working/wheels")]
    input_root = Path("/kaggle/input")
    if input_root.exists():
        for child in input_root.iterdir():
            if child.is_dir():
                candidates.extend([child / "wheels", child])
                for grandchild in child.iterdir():
                    if grandchild.is_dir():
                        candidates.extend([grandchild / "wheels", grandchild])
    out = []
    seen = set()
    for c in candidates:
        c = c.expanduser()
        if c not in seen:
            seen.add(c)
            if _has_package_file(c):
                out.append(c)
    return out


def purge_imported_modules(package_names):
    roots = {"tracksdata"}
    for name in package_names:
        if name in PACKAGE_SPECS:
            roots.add(PACKAGE_SPECS[name][0].split(".")[0])
        if name == "polars":
            roots.add("polars")
    for root in roots:
        for mod in list(sys.modules):
            if mod == root or mod.startswith(root + "."):
                sys.modules.pop(mod, None)


def polars_runtime_ready():
    try:
        import polars as _pl
        from polars._plr import PySeries as _PS
        _ = _PS
        return hasattr(_pl, "Float16") and _pl.Series([-999999.0], dtype=_pl.Float64).dtype == _pl.Float64
    except Exception:
        return False


def packages_requiring_refresh():
    refresh = []
    if not module_missing("polars") and not polars_runtime_ready():
        refresh.append("polars")
    if not module_missing("zarr"):
        try:
            import zarr as _z
            if int(str(getattr(_z, "__version__", "0")).split(".")[0]) < 3:
                refresh.append("zarr")
        except Exception:
            refresh.append("zarr")
    return refresh


def dependency_specs_for(missing):
    specs = []
    seen = set()
    def add(spec):
        if spec.lower() not in seen:
            seen.add(spec.lower()); specs.append(spec)
    for name in missing:
        if name in PACKAGE_SPECS:
            add(PACKAGE_SPECS[name][1])
        for spec in EXTRA_SPECS_BY_NAME.get(name, []):
            add(spec)
    return specs


def import_failures():
    failures = {}
    for name, module_name in REQUIRED_MODULES.items():
        try:
            importlib.import_module(module_name)
        except Exception as exc:
            failures[name] = f"{type(exc).__name__}: {exc}"
    return failures


def missing_names_from_failures(failures):
    names = []
    module_to_name = {module: name for name, module in REQUIRED_MODULES.items()}
    for message in failures.values():
        m = re.search(r"No module named ['\"]([^'\"]+)['\"]", message)
        if m:
            module = m.group(1).split(".")[0]
        else:
            m = re.search(r"module ['\"]([^'\"]+)['\"] has no attribute", message)
            if not m:
                continue
            module = m.group(1).split(".")[0]
        name = module_to_name.get(module)
        if name and name not in names:
            names.append(name)
    return names


def install_missing_dependencies(missing, artifacts):
    specs = dependency_specs_for(missing)
    force_reinstall = bool({"polars", "zarr"} & set(missing))
    if not specs:
        return
    package_dirs = find_offline_package_dirs(artifacts)
    if package_dirs:
        cmd = [sys.executable, "-m", "pip", "install", "--no-index", "--no-deps"]
        if force_reinstall:
            cmd.append("--force-reinstall")
        for pd_ in package_dirs:
            cmd.extend(["--find-links", str(pd_)])
        cmd.extend(specs)
        print("Installing offline:", missing)
        r = subprocess.run(cmd, text=True, capture_output=True)
        if r.returncode == 0:
            purge_imported_modules(missing)
            print("Offline install succeeded.")
            return
        print("Offline failed:", (r.stderr or "")[-1000:])
    if ALLOW_PIP_INSTALL:
        cmd = [sys.executable, "-m", "pip", "install", "--no-deps"]
        if force_reinstall:
            cmd.append("--force-reinstall")
        cmd.extend(specs)
        r = subprocess.run(cmd, text=True, capture_output=True)
        if r.returncode == 0:
            purge_imported_modules(missing)
            print("PyPI install succeeded.")
            return
    raise ImportError("Missing packages: " + ", ".join(missing))


def ensure_dependencies(artifacts):
    for _ in range(5):
        refresh = packages_requiring_refresh()
        if refresh:
            install_missing_dependencies(refresh, artifacts); continue
        missing = [p for p, m in REQUIRED_MODULES.items() if module_missing(m)]
        if missing:
            install_missing_dependencies(missing, artifacts); continue
        failures = import_failures()
        if not failures:
            print("All required packages import successfully."); return
        missing_from_import = missing_names_from_failures(failures)
        if missing_from_import:
            install_missing_dependencies(missing_from_import, artifacts); continue
        raise ImportError("Import failures:\n" + json.dumps(failures, indent=2))
    raise ImportError("Dependency recovery did not converge.")


def remove_path(path):
    if path.is_symlink() or path.is_file():
        path.unlink()
    elif path.exists():
        shutil.rmtree(path)


def copy_or_extract_tree(src_dir, src_zip, dst):
    remove_path(dst)
    if src_dir.exists() and src_dir.is_dir():
        shutil.copytree(src_dir, dst); return
    if src_zip.exists() and src_zip.is_file():
        dst.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(src_zip) as zf:
            zf.extractall(dst)
        return
    raise FileNotFoundError(f"Missing source: {src_dir} / {src_zip}")


def link_or_copy_tree(src, dst):
    remove_path(dst)
    try:
        os.symlink(src, dst, target_is_directory=True)
    except Exception:
        shutil.copytree(src, dst)


def materialize_inference_repo(artifacts):
    copy_or_extract_tree(artifacts / "repo", artifacts / "repo.zip", REPO_DIR)
    weights_src = artifacts / "weights"
    weights_zip = artifacts / "weights.zip"
    weights_dst = REPO_DIR / "weights"
    if weights_src.exists():
        link_or_copy_tree(weights_src, weights_dst)
    elif weights_zip.exists():
        remove_path(weights_dst)
        weights_dst.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(weights_zip) as zf:
            zf.extractall(weights_dst)
    else:
        raise FileNotFoundError(f"Missing weights under {artifacts}")
    required = [REPO_DIR / "scripts" / "predict_unet_transformer.py",
                REPO_DIR / WEIGHTS_RELATIVE]
    missing = [str(p) for p in required if not p.exists()]
    if missing:
        raise FileNotFoundError("Incomplete repo:\n" + "\n".join(missing))
    print("Inference repo:", REPO_DIR)
    print("Weights:", REPO_DIR / WEIGHTS_RELATIVE)


ARTIFACTS = find_artifacts_root()
print("ARTIFACTS:", ARTIFACTS)
print("Has offline wheels:", (ARTIFACTS / "wheels").exists())
manifest_info = artifact_manifest(ARTIFACTS)
if manifest_info:
    print("Artifact name:", manifest_info.get("artifact_name"))
    print("Weight sha256:", manifest_info.get("model", {}).get("weight_sha256"))

ensure_dependencies(ARTIFACTS)
materialize_inference_repo(ARTIFACTS)


ARTIFACTS: /kaggle/input/datasets/pilkwang/biohub-tracking-support-pack-50ep-v1
Has offline wheels: True
Artifact name: biohub-tracking-support-pack-159ep-snapshot-v1
Weight sha256: ff218aba1af34769bba820bb95867052b067a5a730925c3f0ea7c6c72b94d0dc
Installing offline: ['polars']
Offline install succeeded.
Installing offline: ['tracksdata', 'zarr', 'pyscipopt', 'geff', 'geff_spec', 'ilpy', 'imagecodecs', 'rustworkx', 'numcodecs', 'donfig', 'bidict']
Offline install succeeded.


/usr/local/lib/python3.12/dist-packages/dask/array/image.py:7: FutureWarning: `find_available_plugins` is deprecated since version 0.25 and will be removed in version 0.27. The plugin infrastructure of `skimage.io` is deprecated. Instead, use `imageio` or other I/O packages directly.
  from skimage.io import imread as sk_imread
/usr/lib/python3.12/importlib/__init__.py:90: FutureWarning: `reset_plugins` is deprecated since version 0.25 and will be removed in version 0.27. The plugin infrastructure of `skimage.io` is deprecated. Instead, use `imageio` or other I/O packages directly.
  return _bootstrap._gcd_import(name[level:], package, level)


All required packages import successfully.
Inference repo: /kaggle/working/tracking_repo
Weights: /kaggle/working/tracking_repo/weights/unet_transformer/split_0/edge_predictor_best.pth


In [3]:
def list_test_stems():
    if not TEST_DIR.exists():
        raise FileNotFoundError(f"Test dir does not exist: {TEST_DIR}")
    stems = sorted(p.name[:-5] for p in TEST_DIR.iterdir() if p.name.endswith(".zarr"))
    if not stems:
        raise FileNotFoundError(f"No .zarr files in {TEST_DIR}")
    return stems


def _replace_arg(cmd, flag, value):
    out = list(cmd)
    if flag not in out:
        out.extend([flag, value]); return out
    idx = out.index(flag)
    out[idx + 1] = value if idx + 1 < len(out) else out.append(value)
    return out


def _clear_prediction_cache():
    p = REPO_DIR / "predictions"
    if p.exists():
        shutil.rmtree(p)


def run_prediction_with_batch_fallback(base_cmd):
    batch_candidates = []
    for c in [UNET_BATCH_SIZE, max(1, UNET_BATCH_SIZE // 2), 1]:
        if c not in batch_candidates:
            batch_candidates.append(c)
    last_error = None
    for batch_size in batch_candidates:
        trial_cmd = _replace_arg(base_cmd, "--unet-batch-size", str(batch_size))
        _clear_prediction_cache()
        print(f"Prediction attempt | batch_size={batch_size} | preset={BIOHUB_PRESET}")
        print(" ".join(trial_cmd))
        start = time.time()
        try:
            subprocess.run(trial_cmd, cwd=REPO_DIR,
                           env={**os.environ, "PYTHONPATH": "src"}, check=True)
            return time.time() - start, batch_size
        except subprocess.CalledProcessError as exc:
            last_error = exc
            print(f"Failed at batch_size={batch_size}; trying smaller batch.")
    raise last_error


test_stems = list_test_stems()
print(f"Found {len(test_stems)} test videos: {test_stems}")

splits_path = REPO_DIR / "kaggle_test_splits_50ep.json"
splits_path.parent.mkdir(parents=True, exist_ok=True)
splits_path.write_text(json.dumps([{"split": 0, "train": [], "test": test_stems}], indent=2))

predict_cmd = [
    sys.executable, "scripts/predict_unet_transformer.py",
    "--data-dir", str(TEST_DIR),
    "--splits", str(splits_path.name),
    "--split", "0",
    "--weights", WEIGHTS_RELATIVE,
    "--unet-batch-size", str(UNET_BATCH_SIZE),
    "--det-threshold", str(DET_THRESHOLD),
    "--ilp-edge-weight", str(ILP_EDGE_WEIGHT),
    "--ilp-appearance-weight", str(ILP_APPEARANCE_WEIGHT),
    "--ilp-disappearance-weight", str(ILP_DISAPPEARANCE_WEIGHT),
    "--ilp-division-weight", str(ILP_DIVISION_WEIGHT),
]
if USE_ILP:
    predict_cmd.append("--use-ilp")
if SLICE:
    predict_cmd.extend(["--slice", SLICE])

predict_seconds, effective_batch_size = run_prediction_with_batch_fallback(predict_cmd)
print(f"Prediction done in {predict_seconds/60:.2f} min | batch_size={effective_batch_size}")


Found 4 test videos: ['44b6_0113de3b', '44b6_0b24845f', '6bba_05b6850b', '6bba_05db0fb1']
Prediction attempt | batch_size=4 | preset=aggressive
/usr/bin/python3 scripts/predict_unet_transformer.py --data-dir /kaggle/input/competitions/biohub-cell-tracking-during-development/test --splits kaggle_test_splits_50ep.json --split 0 --weights weights/unet_transformer/split_0/edge_predictor_best.pth --unet-batch-size 4 --det-threshold 0.985 --ilp-edge-weight -1.0 --ilp-appearance-weight 0.1 --ilp-disappearance-weight 0.1 --ilp-division-weight 0.7 --use-ilp


/usr/local/lib/python3.12/dist-packages/dask/array/image.py:7: FutureWarning: `find_available_plugins` is deprecated since version 0.25 and will be removed in version 0.27. The plugin infrastructure of `skimage.io` is deprecated. Instead, use `imageio` or other I/O packages directly.
  from skimage.io import imread as sk_imread
/usr/lib/python3.12/importlib/__init__.py:90: FutureWarning: `reset_plugins` is deprecated since version 0.25 and will be removed in version 0.27. The plugin infrastructure of `skimage.io` is deprecated. Instead, use `imageio` or other I/O packages directly.
  return _bootstrap._gcd_import(name[level:], package, level)


Fold 0: 4 datasets | weights=weights/unet_transformer/split_0/edge_predictor_best.pth | device=cuda | window_size=2 | pool_kernel_um=3.0
Saved 4 predictions to /kaggle/working/tracking_repo/predictions/unknown/unet_transformer/split_0
Prediction done in 6.31 min | batch_size=4


In [4]:
import tracksdata as td
import numpy as np
import blosc2
from scipy.optimize import linear_sum_assignment

SUBMISSION_COLUMNS = ["dataset","row_type","node_id","t","z","y","x","source_id","target_id"]
CSV_COLUMNS = ["id", *SUBMISSION_COLUMNS]
VOXEL_SCALE_UM = (1.625, 0.40625, 0.40625)


def graph_from_geff(path):
    graph = td.graph.IndexedRXGraph.from_geff(path)
    return graph[0] if isinstance(graph, tuple) else graph


def edge_distance_um(source, target):
    dz = (float(source["z"]) - float(target["z"])) * VOXEL_SCALE_UM[0]
    dy = (float(source["y"]) - float(target["y"])) * VOXEL_SCALE_UM[1]
    dx = (float(source["x"]) - float(target["x"])) * VOXEL_SCALE_UM[2]
    return math.sqrt(dz*dz + dy*dy + dx*dx)


def point_distance_um(a, b):
    dz = (a[0]-b[0])*VOXEL_SCALE_UM[0]
    dy = (a[1]-b[1])*VOXEL_SCALE_UM[1]
    dx = (a[2]-b[2])*VOXEL_SCALE_UM[2]
    return math.sqrt(dz*dz + dy*dy + dx*dx)


def node_point(node):
    return (float(node["z"]), float(node["y"]), float(node["x"]))


def edge_sort_key(edge):
    prob = edge.get("edge_prob")
    prob_value = float(prob) if prob is not None else 0.0
    return prob_value, -float(edge["distance_um"])


def _next_node_id(nodes_by_id):
    return max(nodes_by_id) + 1 if nodes_by_id else 1


def read_test_frame(dataset, t, frame_cache):
    if t in frame_cache:
        return frame_cache[t]
    zarr_path = TEST_DIR / f"{dataset}.zarr"
    meta = json.loads((zarr_path/"0"/"zarr.json").read_text())
    shape = tuple(int(v) for v in meta["shape"])
    dtype = np.dtype(meta["data_type"])
    frame_shape = shape[1:]
    chunk_path = zarr_path/"0"/"c"/str(t)/"0"/"0"/"0"
    try:
        raw = chunk_path.read_bytes()
        arr = np.frombuffer(blosc2.decompress(raw), dtype=dtype)
        if arr.size == int(np.prod(frame_shape)):
            frame = arr.reshape(frame_shape).copy()
            frame_cache[t] = frame
            return frame
    except Exception:
        pass
    import zarr
    frame = np.asarray(zarr.open(zarr_path/"0", mode="r")[t])
    frame_cache[t] = frame
    return frame


def refine_synthetic_midpoint(dataset, t, midpoint, frame_cache, stats):
    if not GAP_REFINE_SYNTHETIC or dataset is None:
        return midpoint
    try:
        frame = read_test_frame(dataset, t, frame_cache)
        z, y, x = [int(round(v)) for v in midpoint]
        z0 = max(0, z-GAP_REFINE_WIN_Z);  z1 = min(frame.shape[0], z+GAP_REFINE_WIN_Z+1)
        y0 = max(0, y-GAP_REFINE_WIN_YX); y1 = min(frame.shape[1], y+GAP_REFINE_WIN_YX+1)
        x0 = max(0, x-GAP_REFINE_WIN_YX); x1 = min(frame.shape[2], x+GAP_REFINE_WIN_YX+1)
        patch = frame[z0:z1, y0:y1, x0:x1].astype(np.float64)
        if patch.size == 0:
            stats["gap_refine_failed"] += 1; return midpoint
        baseline = float(np.percentile(patch, 20.0))
        weights = np.maximum(patch - baseline, 0.0)
        total = float(weights.sum())
        if total <= 0:
            stats["gap_refine_failed"] += 1; return midpoint
        zz = np.arange(z0,z1,dtype=np.float64)[:,None,None]
        yy = np.arange(y0,y1,dtype=np.float64)[None,:,None]
        xx = np.arange(x0,x1,dtype=np.float64)[None,None,:]
        refined = (float((weights*zz).sum()/total),
                   float((weights*yy).sum()/total),
                   float((weights*xx).sum()/total))
        if point_distance_um(refined, midpoint) > GAP_REFINE_MAX_SHIFT_UM:
            stats["gap_refine_rejected_shift"] += 1; return midpoint
        stats["gap_refined_synthetic"] += 1
        return refined
    except Exception:
        stats["gap_refine_failed"] += 1; return midpoint


def _position_um(node):
    return np.array([float(node["z"])*VOXEL_SCALE_UM[0],
                     float(node["y"])*VOXEL_SCALE_UM[1],
                     float(node["x"])*VOXEL_SCALE_UM[2]], dtype=np.float64)


def motion_relink_edges(nodes_by_id, stats, learned_edge_probs=None):
    if not OUTPUT_MOTION_RELINK or not nodes_by_id:
        return []
    learned_edge_probs = learned_edge_probs or {}

    def learned_prob(sid, tid):
        v = learned_edge_probs.get((sid, tid), 0.0)
        try: v = float(v)
        except: return 0.0
        if not np.isfinite(v): return 0.0
        if v < 0 or v > 1:
            v = 1.0/(1.0+math.exp(-max(-20., min(20., v))))
        return float(np.clip(v, 0.0, 1.0))

    ids_by_t = {}
    for nid, node in nodes_by_id.items():
        ids_by_t.setdefault(int(node["t"]), []).append(nid)
    for ids in ids_by_t.values():
        ids.sort()

    frame_sizes = [len(v) for v in ids_by_t.values()]
    if frame_sizes and max(frame_sizes) > MOTION_RELINK_MAX_FRAME_NODES:
        stats["motion_relink_skipped_large_frame"] = 1; return []

    position_um = {nid: _position_um(node) for nid, node in nodes_by_id.items()}
    predecessor_position_um = {}
    selected_edges = []

    def assign_pass(source_ids, target_ids, gate_um):
        if not source_ids or not target_ids: return []
        big = gate_um*1000+1
        cost = np.full((len(source_ids), len(target_ids)), big, dtype=np.float64)
        raw_dist = np.full_like(cost, np.inf)
        motion_dist = np.full_like(cost, np.inf)
        prob_matrix = np.zeros_like(cost)
        for i, sid in enumerate(source_ids):
            sp = position_um[sid]
            pp = predecessor_position_um.get(sid)
            predicted = sp + MOTION_RELINK_VELOCITY_WEIGHT*(sp-pp) if pp is not None else sp
            for j, tid in enumerate(target_ids):
                tp = position_um[tid]
                raw = float(np.linalg.norm(tp-sp))
                if raw > gate_um: continue
                motion = float(np.linalg.norm(tp-predicted))
                prob = learned_prob(sid, tid)
                raw_dist[i,j]=raw; motion_dist[i,j]=motion; prob_matrix[i,j]=prob
                cost[i,j] = motion + 0.05*raw - MOTION_RELINK_LEARNED_BONUS*prob
        ri, ci = linear_sum_assignment(cost)
        return [(source_ids[int(r)], target_ids[int(c)],
                 float(raw_dist[r,c]), float(motion_dist[r,c]), float(prob_matrix[r,c]))
                for r,c in zip(ri,ci) if cost[r,c] < big]

    for t in sorted(ids_by_t):
        src_ids = ids_by_t.get(t, [])
        tgt_ids = ids_by_t.get(t+1, [])
        if not src_ids or not tgt_ids: continue
        unmatched_s = set(src_ids); unmatched_t = set(tgt_ids)
        frame_matches = []
        for pass_name, gate_um in (("tight", MOTION_RELINK_TIGHT_UM), ("relaxed", MOTION_RELINK_RELAXED_UM)):
            ps = [n for n in src_ids if n in unmatched_s]
            pt = [n for n in tgt_ids if n in unmatched_t]
            for sid, tid, raw, motion, prob in assign_pass(ps, pt, gate_um):
                if sid not in unmatched_s or tid not in unmatched_t: continue
                unmatched_s.remove(sid); unmatched_t.remove(tid)
                frame_matches.append((sid, tid, raw, motion, pass_name, prob))
                stats["motion_relink_tight_edges" if pass_name=="tight" else "motion_relink_relaxed_edges"] += 1
        for sid, tid, raw, motion, pass_name, prob in frame_matches:
            selected_edges.append({"source_id":sid,"target_id":tid,"edge_prob":prob,
                                    "distance_um":raw,"motion_distance_um":motion,
                                    "motion_relinked":1,"motion_pass":pass_name})
            predecessor_position_um[tid] = position_um[sid]
        stats["motion_relink_frames"] += 1

    stats["motion_relink_edges"] = len(selected_edges)
    return selected_edges


def close_single_frame_gaps(nodes_by_id, edges, stats, dataset=None):
    if not OUTPUT_GAP_CLOSE or GAP_CLOSE_MAX_GAP < 1 or not edges:
        return nodes_by_id, edges
    outgoing = {int(e["source_id"]) for e in edges}
    incoming = {int(e["target_id"]) for e in edges}
    incident = outgoing | incoming
    ends_by_t = {}; starts_by_t = {}; isolated_by_t = {}
    for nid, node in nodes_by_id.items():
        t = int(node["t"])
        if nid not in outgoing: ends_by_t.setdefault(t, []).append(nid)
        if nid not in incoming: starts_by_t.setdefault(t, []).append(nid)
        if nid not in incident: isolated_by_t.setdefault(t, []).append(nid)
    max_synthetic = min(GAP_CLOSE_MAX_ADDED_ABS,
                        max(1, int(round(len(nodes_by_id)*GAP_CLOSE_MAX_ADDED_FRAC)))
                        if GAP_CLOSE_MAX_ADDED_FRAC > 0 else 0)
    next_id = _next_node_id(nodes_by_id)
    frame_cache = {}; used_starts = set(); used_isolated = set()
    synthetic_added = 0; new_edges = []
    effective_gap_max = min(GAP_CLOSE_MAX_GAP, 1)
    stats["gap_close_effective_max_gap"] = effective_gap_max
    for gap in range(1, effective_gap_max+1):
        for t, end_ids in sorted(ends_by_t.items()):
            start_ids = [s for s in starts_by_t.get(t+gap+1, []) if s not in used_starts]
            if not end_ids or not start_ids: continue
            end_points   = [node_point(nodes_by_id[e]) for e in end_ids]
            start_points = [node_point(nodes_by_id[s]) for s in start_ids]
            threshold_um = GAP_CLOSE_UM*(gap+1)
            d = np.zeros((len(end_ids), len(start_ids)))
            for i,ep in enumerate(end_points):
                for j,sp in enumerate(start_points):
                    d[i,j] = point_distance_um(ep,sp)
            stats["gap_candidates"] += int((d<=threshold_um).sum())
            big = threshold_um*1000+1
            cost = np.where(d<=threshold_um, d, big)
            ri, ci = linear_sum_assignment(cost)
            for r,c in zip(ri,ci):
                if d[r,c]>threshold_um: continue
                sid=end_ids[int(r)]; tid=start_ids[int(c)]
                if sid in outgoing or tid in used_starts: continue
                source=nodes_by_id[sid]; target=nodes_by_id[tid]
                mid_t=int(source["t"])+gap
                mid_point=((float(source["z"])+float(target["z"]))/2,
                            (float(source["y"])+float(target["y"]))/2,
                            (float(source["x"])+float(target["x"]))/2)
                middle_id = None
                if GAP_CLOSE_REUSE_EXISTING:
                    cands = [n for n in isolated_by_t.get(mid_t,[]) if n not in used_isolated]
                    if cands:
                        dists = [point_distance_um(node_point(nodes_by_id[n]), mid_point) for n in cands]
                        bi = int(np.argmin(dists))
                        if dists[bi] <= GAP_CLOSE_REUSE_UM:
                            middle_id=cands[bi]; used_isolated.add(middle_id)
                            stats["gap_reused_existing"] += 1
                if middle_id is None:
                    if synthetic_added >= max_synthetic:
                        stats["gap_skipped_node_cap"] += 1; continue
                    middle_id = next_id; next_id += 1
                    rp = refine_synthetic_midpoint(dataset, mid_t, mid_point, frame_cache, stats)
                    nodes_by_id[middle_id] = {"node_id":middle_id,"t":mid_t,"z":rp[0],"y":rp[1],"x":rp[2]}
                    synthetic_added += 1; stats["gap_inserted_synthetic"] += 1
                middle = nodes_by_id[middle_id]
                new_edges.extend([
                    {"source_id":sid,"target_id":middle_id,"edge_prob":None,
                     "distance_um":edge_distance_um(source,middle),"gap_closed":1},
                    {"source_id":middle_id,"target_id":tid,"edge_prob":None,
                     "distance_um":edge_distance_um(middle,target),"gap_closed":1},
                ])
                outgoing.add(sid); incoming.add(middle_id)
                outgoing.add(middle_id); incoming.add(tid)
                used_starts.add(tid)
                stats["gap_pairs_selected"] += 1; stats["gap_added_edges"] += 2
    if new_edges:
        edges = [*edges, *new_edges]
    stats["gap_added_nodes"] = stats["gap_inserted_synthetic"]
    return nodes_by_id, edges


def _single_successor_map(edges):
    by_source = {}
    for e in edges:
        by_source.setdefault(int(e["source_id"]), []).append(int(e["target_id"]))
    return {s: t[0] for s,t in by_source.items() if len(t)==1}


def _single_predecessor_map(edges):
    by_target = {}
    for e in edges:
        by_target.setdefault(int(e["target_id"]), []).append(int(e["source_id"]))
    return {t: s[0] for t,s in by_target.items() if len(s)==1}


def recover_strict_gap2(nodes_by_id, edges, stats, dataset=None):
    if not OUTPUT_GAP2_RECOVERY or not edges or not nodes_by_id:
        return nodes_by_id, edges
    outgoing  = {int(e["source_id"]) for e in edges}
    incoming  = {int(e["target_id"]) for e in edges}
    predecessor = _single_predecessor_map(edges)
    successor   = _single_successor_map(edges)
    ends_by_t = {}; starts_by_t = {}
    for nid, node in nodes_by_id.items():
        t = int(node["t"])
        if nid not in outgoing:  ends_by_t.setdefault(t, []).append(nid)
        if nid not in incoming: starts_by_t.setdefault(t, []).append(nid)
    cap = min(GAP2_MAX_LINKS_ABS, max(1, int(round(len(edges)*GAP2_MAX_LINKS_FRAC))))
    def pos_um(nid):
        n = nodes_by_id[nid]
        return np.array([float(n["z"]),float(n["y"]),float(n["x"])],dtype=np.float64)*np.array(VOXEL_SCALE_UM)
    proposals = []
    for t, end_ids in sorted(ends_by_t.items()):
        for end_id in end_ids:
            ep = pos_um(end_id)
            for start_id in starts_by_t.get(t+3, []):
                sp = pos_um(start_id)
                dist = float(np.linalg.norm(sp-ep))
                if dist>GAP2_MAX_TOTAL_UM or dist/3>GAP2_MAX_STEP_UM: continue
                step=(sp-ep)/3; ctx_pen=0.0
                if GAP2_REQUIRE_CONTEXT:
                    ok=False
                    prev_id=predecessor.get(end_id)
                    if prev_id is not None:
                        ps=ep-pos_um(prev_id); pn=float(np.linalg.norm(ps)); sn=float(np.linalg.norm(step))
                        if pn<=0.01 or sn<=0.01: ok=True
                        else:
                            cos=float(np.dot(ps,step)/(pn*sn+1e-9))
                            if cos>-0.25 and np.linalg.norm(ps-step)<=6: ok=True
                            ctx_pen+=max(0,0.25-cos)
                    next_id=successor.get(start_id)
                    if next_id is not None:
                        ns=pos_um(next_id)-sp; nn=float(np.linalg.norm(ns)); sn=float(np.linalg.norm(step))
                        if nn<=0.01 or sn<=0.01: ok=True
                        else:
                            cos=float(np.dot(ns,step)/(nn*sn+1e-9))
                            if cos>-0.25 and np.linalg.norm(ns-step)<=6: ok=True
                            ctx_pen+=max(0,0.25-cos)
                    if not ok: continue
                proposals.append((dist+2*ctx_pen, end_id, start_id, t, dist))
    proposals.sort(key=lambda x: x[0])
    stats["gap2_candidates"] = len(proposals)
    if not proposals: return nodes_by_id, edges
    selected=[]; used_ends=set(); used_starts=set(); per_frame={}
    for prop in proposals:
        if len(selected)>=cap: stats["gap2_skipped_cap"]+=1; break
        _, end_id, start_id, t, _ = prop
        if end_id in used_ends or start_id in used_starts: continue
        fcap=max(1,int(round(len(ends_by_t.get(t,[]))*GAP2_FRAME_FRAC_CAP)))
        if per_frame.get(t,0)>=fcap: continue
        selected.append(prop); used_ends.add(end_id); used_starts.add(start_id)
        per_frame[t]=per_frame.get(t,0)+1
    if not selected: return nodes_by_id, edges
    next_nid=_next_node_id(nodes_by_id); frame_cache={}; new_edges=[]
    for _, end_id, start_id, t, _ in selected:
        source=nodes_by_id[end_id]; target=nodes_by_id[start_id]
        prev_id=end_id
        for k in (1,2):
            frac=k/3; mid_t=int(source["t"])+k
            mp=(float(source["z"])+(float(target["z"])-float(source["z"]))*frac,
                float(source["y"])+(float(target["y"])-float(source["y"]))*frac,
                float(source["x"])+(float(target["x"])-float(source["x"]))*frac)
            rp=refine_synthetic_midpoint(dataset, mid_t, mp, frame_cache, stats)
            nid=next_nid; next_nid+=1
            nodes_by_id[nid]={"node_id":nid,"t":mid_t,"z":rp[0],"y":rp[1],"x":rp[2]}
            new_edges.append({"source_id":prev_id,"target_id":nid,"edge_prob":None,
                              "distance_um":edge_distance_um(nodes_by_id[prev_id],nodes_by_id[nid]),"gap2_recovered":1})
            prev_id=nid
        new_edges.append({"source_id":prev_id,"target_id":start_id,"edge_prob":None,
                          "distance_um":edge_distance_um(nodes_by_id[prev_id],target),"gap2_recovered":1})
        stats["gap2_pairs_selected"]+=1; stats["gap2_added_nodes"]+=2; stats["gap2_added_edges"]+=3
    return nodes_by_id, [*edges, *new_edges]


def add_safe_divisions_postlink(nodes_by_id, edges, stats):
    if not OUTPUT_SAFE_DIVISIONS or not edges or not nodes_by_id:
        return edges
    out_by_source={}; incoming=set()
    for e in edges:
        out_by_source.setdefault(int(e["source_id"]),[]).append(e)
        incoming.add(int(e["target_id"]))
    ids_by_t={}
    for nid,node in nodes_by_id.items():
        ids_by_t.setdefault(int(node["t"]),[]).append(nid)
    existing_edges={(int(e["source_id"]),int(e["target_id"])) for e in edges}
    global_cap=max(1,int(round(max(1,len(edges))*SAFE_DIV_GLOBAL_FRAC_CAP)))
    added=[]; used_targets=set()
    for t in sorted(ids_by_t):
        child_frame_ids=ids_by_t.get(t+1,[])
        if not child_frame_ids: continue
        source_ids=[n for n in ids_by_t[t] if len(out_by_source.get(n,[]))==1]
        candidate_ids=[n for n in child_frame_ids if n not in incoming and n not in used_targets]
        if not source_ids or not candidate_ids: continue
        frame_cap=max(1,int(round(len(source_ids)*SAFE_DIV_FRAME_FRAC_CAP)))
        proposals=[]
        for sid in source_ids:
            source=nodes_by_id[sid]
            ec_edge=out_by_source[sid][0]
            ec_id=int(ec_edge["target_id"])
            ec=nodes_by_id.get(ec_id)
            if ec is None or int(ec["t"])!=t+1: continue
            if edge_distance_um(source,ec)>SAFE_DIV_EXISTING_CHILD_MAX_UM: continue
            for cid in candidate_ids:
                if (sid,cid) in existing_edges: continue
                cand=nodes_by_id[cid]
                pd_=edge_distance_um(source,cand)
                if pd_>SAFE_DIV_MAX_UM: continue
                sd_=edge_distance_um(ec,cand)
                if sd_>SAFE_DIV_SISTER_MAX_UM: continue
                proposals.append((pd_+0.15*sd_, sid, cid, pd_))
        stats["safe_division_candidates"]+=len(proposals)
        if not proposals: continue
        proposals.sort(key=lambda x:x[0])
        added_this_frame=0
        for _,sid,cid,pd_ in proposals:
            if len(added)>=global_cap: stats["safe_division_skipped_cap"]+=1; break
            if added_this_frame>=frame_cap: break
            if cid in used_targets or cid in incoming: continue
            cand=nodes_by_id[cid]
            added.append({"source_id":sid,"target_id":cid,"edge_prob":None,"distance_um":pd_,"safe_division":1})
            used_targets.add(cid); added_this_frame+=1
    if added:
        stats["safe_divisions_added"]=len(added)
        return [*edges, *added]
    return edges


def filter_short_track_components(nodes_by_id, edges, stats):
    if not OUTPUT_FILTER_SHORT_TRACKS or OUTPUT_MIN_TRACK_LEN<=1 or not edges:
        return nodes_by_id, edges
    parent={nid:nid for nid in nodes_by_id}
    def find(nid):
        while parent[nid]!=nid: parent[nid]=parent[parent[nid]]; nid=parent[nid]
        return nid
    def union(a,b):
        if a not in parent or b not in parent: return
        ra=find(a); rb=find(b)
        if ra!=rb: parent[ra]=rb
    out_count={}
    for e in edges:
        sid=int(e["source_id"]); tid=int(e["target_id"])
        union(sid,tid); out_count[sid]=out_count.get(sid,0)+1
    components={}
    for nid in nodes_by_id: components.setdefault(find(nid),[]).append(nid)
    keep=set()
    for members in components.values():
        has_div=any(out_count.get(n,0)>=2 for n in members)
        if len(members)>=OUTPUT_MIN_TRACK_LEN or (OUTPUT_KEEP_DIVISION_COMPONENTS and has_div):
            keep.update(members)
    if not keep: stats["short_track_filter_skipped_all"]+=1; return nodes_by_id, edges
    removed=len(nodes_by_id)-len(keep)
    if removed<=0: return nodes_by_id, edges
    kept_nodes={n:node for n,node in nodes_by_id.items() if n in keep}
    kept_edges=[e for e in edges if int(e["source_id"]) in kept_nodes and int(e["target_id"]) in kept_nodes]
    stats["short_track_nodes_removed"]=removed
    stats["short_track_edges_removed"]=len(edges)-len(kept_edges)
    return kept_nodes, kept_edges


def linefit_smooth_output_graph(nodes_by_id, edges, stats):
    if not OUTPUT_LINEFIT_SMOOTH or OUTPUT_LINEFIT_WEIGHT<=0 or OUTPUT_LINEFIT_WINDOW<=0 or not edges:
        return nodes_by_id
    predecessor={}; successor={}
    for e in edges:
        sid=int(e["source_id"]); tid=int(e["target_id"])
        sn=nodes_by_id.get(sid); tn=nodes_by_id.get(tid)
        if sn is None or tn is None: continue
        if int(tn["t"])!=int(sn["t"])+1: continue
        successor.setdefault(sid,[]).append(tid)
        predecessor.setdefault(tid,[]).append(sid)
    orig={nid:np.array([float(n["z"]),float(n["y"]),float(n["x"])],dtype=np.float64)
          for nid,n in nodes_by_id.items()}
    updated={}; weight=float(np.clip(OUTPUT_LINEFIT_WEIGHT, 0, 1))
    for nid in sorted(nodes_by_id):
        nbhd=[(0,nid)]
        cur=nid
        for step in range(1,OUTPUT_LINEFIT_WINDOW+1):
            pids=predecessor.get(cur,[])
            if len(pids)!=1: break
            cur=pids[0]
            if cur not in orig: break
            nbhd.append((-step,cur))
        cur=nid
        for step in range(1,OUTPUT_LINEFIT_WINDOW+1):
            nids_=successor.get(cur,[])
            if len(nids_)!=1: break
            cur=nids_[0]
            if cur not in orig: break
            nbhd.append((step,cur))
        if len(nbhd)<3: stats["linefit_skipped_nodes"]+=1; continue
        dts=np.array([d for d,_ in nbhd],dtype=np.float64)
        coords=np.stack([orig[n] for _,n in nbhd])
        fitted=np.array([np.polyval(np.polyfit(dts,coords[:,ax],1),0.) for ax in range(3)],dtype=np.float64)
        if not np.isfinite(fitted).all(): stats["linefit_skipped_nodes"]+=1; continue
        updated[nid]=(1-weight)*orig[nid]+weight*fitted
    for nid,pos in updated.items():
        nodes_by_id[nid]["z"]=float(pos[0])
        nodes_by_id[nid]["y"]=float(pos[1])
        nodes_by_id[nid]["x"]=float(pos[2])
    stats["linefit_smoothed_nodes"]=len(updated)
    return nodes_by_id


def filter_output_graph(nodes_by_id, raw_edges, dataset=None):
    stats={
        "raw_edges":len(raw_edges),"dropped_nonconsecutive_edges":0,"dropped_long_edges":0,
        "dropped_multi_parent_edges":0,"dropped_multi_child_edges":0,"dropped_division_edges":0,
        "gap_candidates":0,"gap_pairs_selected":0,"gap_reused_existing":0,
        "gap_inserted_synthetic":0,"gap_added_nodes":0,"gap_added_edges":0,
        "gap_skipped_node_cap":0,"gap_refined_synthetic":0,"gap_refine_failed":0,
        "gap_refine_rejected_shift":0,"pruned_isolated_nodes":0,"motion_relink_edges":0,
        "motion_relink_tight_edges":0,"motion_relink_relaxed_edges":0,"motion_relink_frames":0,
        "motion_relink_replaced_raw_edges":0,"motion_relink_fallback_raw":0,
        "motion_relink_skipped_large_frame":0,"gap2_candidates":0,"gap2_pairs_selected":0,
        "gap2_added_nodes":0,"gap2_added_edges":0,"gap2_skipped_cap":0,
        "safe_division_candidates":0,"safe_divisions_added":0,"safe_division_skipped_cap":0,
        "short_track_components_removed":0,"short_track_nodes_removed":0,
        "short_track_edges_removed":0,"short_track_filter_skipped_all":0,
        "linefit_smoothed_nodes":0,"linefit_skipped_nodes":0,
    }
    edges=[]
    for e in raw_edges:
        source=nodes_by_id.get(int(e["source_id"])); target=nodes_by_id.get(int(e["target_id"]))
        if source is None or target is None: continue
        if OUTPUT_ENFORCE_NEXT_FRAME and int(target["t"])!=int(source["t"])+1:
            stats["dropped_nonconsecutive_edges"]+=1; continue
        dist=edge_distance_um(source,target); e["distance_um"]=dist
        if OUTPUT_EDGE_MAX_UM>0 and dist>OUTPUT_EDGE_MAX_UM:
            stats["dropped_long_edges"]+=1; continue
        edges.append(e)
    if OUTPUT_MOTION_RELINK:
        lep={}
        for e in edges:
            prob=e.get("edge_prob")
            if prob is None: continue
            try: prob=float(prob)
            except: continue
            if np.isfinite(prob):
                k=(int(e["source_id"]),int(e["target_id"]))
                lep[k]=max(lep.get(k,float("-inf")),prob)
        me=motion_relink_edges(nodes_by_id, stats, lep)
        if me:
            stats["motion_relink_replaced_raw_edges"]=len(edges); edges=me
        else:
            stats["motion_relink_fallback_raw"]=1
    if OUTPUT_SINGLE_PARENT_REPAIR and edges:
        best_by_target={}
        for e in edges:
            tid=int(e["target_id"])
            prev=best_by_target.get(tid)
            if prev is None or edge_sort_key(e)>edge_sort_key(prev):
                best_by_target[tid]=e
        kept_ids={id(e) for e in best_by_target.values()}
        stats["dropped_multi_parent_edges"]=sum(1 for e in edges if id(e) not in kept_ids)
        edges=[e for e in edges if id(e) in kept_ids]
    if OUTPUT_SINGLE_CHILD_REPAIR and edges:
        best_by_source={}
        for e in edges:
            sid=int(e["source_id"])
            prev=best_by_source.get(sid)
            if prev is None or edge_sort_key(e)>edge_sort_key(prev):
                best_by_source[sid]=e
        kept_ids={id(e) for e in best_by_source.values()}
        stats["dropped_multi_child_edges"]=sum(1 for e in edges if id(e) not in kept_ids)
        edges=[e for e in edges if id(e) in kept_ids]
    nodes_by_id, edges = close_single_frame_gaps(nodes_by_id, edges, stats, dataset=dataset)
    nodes_by_id, edges = recover_strict_gap2(nodes_by_id, edges, stats, dataset=dataset)
    edges = add_safe_divisions_postlink(nodes_by_id, edges, stats)
    if OUTPUT_DIVISION_GEOMETRY_FILTER and edges:
        by_source={}
        for e in edges: by_source.setdefault(int(e["source_id"]),[]).append(e)
        filtered=[]
        for sid, ses in by_source.items():
            if len(ses)<=1: filtered.extend(ses); continue
            ranked=sorted(ses,key=edge_sort_key,reverse=True)
            source=nodes_by_id[sid]; t1=ranked[0]; t2=ranked[1]
            d1=float(t1["distance_um"]); d2=float(t2["distance_um"])
            sister=edge_distance_um(nodes_by_id[int(t1["target_id"])],nodes_by_id[int(t2["target_id"])])
            valid=(max(d1,d2)<=DIV_PARENT_MAX_UM and sister<=DIV_SISTER_MAX_UM
                   and int(nodes_by_id[int(t1["target_id"])]["t"])==int(source["t"])+1
                   and int(nodes_by_id[int(t2["target_id"])]["t"])==int(source["t"])+1)
            if valid: filtered.extend([t1,t2]); stats["dropped_division_edges"]+=max(0,len(ranked)-2)
            elif DIV_DROP_TO_SINGLE_IF_BAD: filtered.append(t1); stats["dropped_division_edges"]+=len(ranked)-1
            else: filtered.extend(ranked)
        edges=filtered
    if OUTPUT_PRUNE_ISOLATED:
        incident={int(e["source_id"]) for e in edges}|{int(e["target_id"]) for e in edges}
        if incident:
            kept={n:node for n,node in nodes_by_id.items() if n in incident}
            stats["pruned_isolated_nodes"]=len(nodes_by_id)-len(kept)
            nodes_by_id=kept
            edges=[e for e in edges if int(e["source_id"]) in nodes_by_id and int(e["target_id"]) in nodes_by_id]
    nodes_by_id, edges = filter_short_track_components(nodes_by_id, edges, stats)
    nodes_by_id = linefit_smooth_output_graph(nodes_by_id, edges, stats)
    return nodes_by_id, edges, stats


In [5]:
geffs = sorted((REPO_DIR/"predictions").glob(f"*/{METHOD}/split_0/*.geff"))
print(f"Found {len(geffs)} prediction graphs")
if len(geffs)!=len(test_stems):
    found={p.stem for p in geffs}
    missing=sorted(set(test_stems)-found)
    raise RuntimeError(f"Expected {len(test_stems)} graphs, found {len(geffs)}. Missing: {missing[:10]}")

stats_rows=[]; seen_datasets=set(); row_id=0; total_nodes=0; total_edges=0

with SUBMISSION_PATH.open("w",newline="") as f:
    writer=csv.DictWriter(f, fieldnames=CSV_COLUMNS)
    writer.writeheader()
    for geff_path in geffs:
        dataset=geff_path.stem
        seen_datasets.add(dataset)
        graph=graph_from_geff(geff_path)
        nodes_by_id={}
        for row in graph.node_attrs().iter_rows(named=True):
            nid=int(row["node_id"])
            nodes_by_id[nid]={"node_id":nid,"t":int(row["t"]),
                              "z":float(row["z"]),"y":float(row["y"]),"x":float(row["x"])}
        raw_edges=[]
        for row in graph.edge_attrs().iter_rows(named=True):
            ep=row.get("edge_prob") if hasattr(row,"get") else None
            raw_edges.append({"source_id":int(row["source_id"]),"target_id":int(row["target_id"]),
                               "edge_prob":None if ep is None else float(ep)})
        raw_node_count=len(nodes_by_id)
        nodes_by_id, edges, fstats = filter_output_graph(nodes_by_id, raw_edges, dataset=dataset)
        if not nodes_by_id:
            raise AssertionError(f"{dataset}: post-processing removed every node")
        for nid in sorted(nodes_by_id):
            node=nodes_by_id[nid]
            writer.writerow({"id":row_id,"dataset":dataset,"row_type":"node",
                             "node_id":int(node["node_id"]),"t":int(node["t"]),
                             "z":int(round(float(node["z"]))),"y":int(round(float(node["y"]))),
                             "x":int(round(float(node["x"]))),"source_id":-1,"target_id":-1})
            row_id+=1
        division_sources={}
        for e in edges:
            sid=int(e["source_id"]); tid=int(e["target_id"])
            if sid not in nodes_by_id or tid not in nodes_by_id:
                raise AssertionError(f"{dataset}: dangling edge after filtering")
            writer.writerow({"id":row_id,"dataset":dataset,"row_type":"edge",
                             "node_id":-1,"t":-1,"z":-1,"y":-1,"x":-1,
                             "source_id":sid,"target_id":tid})
            row_id+=1
            division_sources[sid]=division_sources.get(sid,0)+1
        nc=len(nodes_by_id); ec=len(edges)
        total_nodes+=nc; total_edges+=ec
        stats_rows.append({"dataset":dataset,"raw_nodes":raw_node_count,"nodes":nc,
                            "raw_edges":fstats["raw_edges"],"edges":ec,
                            "division_like_sources":sum(1 for c in division_sources.values() if c>=2),
                            "edge_to_node_ratio":ec/max(nc,1),
                            "gap_added_nodes_frac":fstats.get("gap_added_nodes",0)/max(raw_node_count,1),
                            **fstats})

missing_datasets=sorted(set(test_stems)-seen_datasets)
if missing_datasets:
    raise AssertionError(f"Missing datasets in output: {missing_datasets}")
assert row_id==total_nodes+total_edges
assert total_nodes>0

stats=pd.DataFrame(stats_rows).sort_values("dataset").reset_index(drop=True)
stats["predict_minutes_total"]=predict_seconds/60.0
stats["experiment_tag"]=EXPERIMENT_TAG
stats["preset"]=BIOHUB_PRESET
stats.to_csv(RUN_STATS_PATH, index=False)

print(f"\nWrote {SUBMISSION_PATH}  ({row_id:,} rows)")
print(f"Node rows: {total_nodes:,}  |  Edge rows: {total_edges:,}")
print(f"Edges/node: {total_edges/max(total_nodes,1):.4f}")

summary_cols=["dataset","raw_nodes","nodes","raw_edges","edges","division_like_sources",
              "gap_added_nodes","gap_added_edges","gap2_added_nodes","gap2_added_edges",
              "safe_divisions_added","linefit_smoothed_nodes","pruned_isolated_nodes"]
summary_cols=[c for c in summary_cols if c in stats.columns]
print("\nPer-dataset summary:")
print(stats[summary_cols].to_string(index=False))
print("\nSubmission preview:")
print(pd.read_csv(SUBMISSION_PATH, nrows=5).to_string(index=False))


Found 4 prediction graphs

Wrote /kaggle/working/submission.csv  (292,390 rows)
Node rows: 149,258  |  Edge rows: 143,132
Edges/node: 0.9590

Per-dataset summary:
      dataset  raw_nodes  nodes  raw_edges  edges  division_like_sources  gap_added_nodes  gap_added_edges  gap2_added_nodes  gap2_added_edges  safe_divisions_added  linefit_smoothed_nodes  pruned_isolated_nodes
44b6_0113de3b      25912  26134      24692  25312                     92              169              338                56                84                    92                   26025                      3
44b6_0b24845f      37503  39236      30749  36533                    218             1617             3234               180               270                   218                   38395                     64
6bba_05b6850b       7289   7427       6642   7075                     36               98              196                42                63                    36                    7313             